# MNIST Test of the Kaplan / McCandlish / Amodei Batch-Size Prediction

## Objective

The goal is to compare the experimentally measured optimal one-step learning rate with the theoretical trend predicted in *An Empirical Model of Large-Batch Training*.

At a fixed checkpoint $\theta_t$, define the standard loss change

$$
\Delta L_B(\epsilon)
=
L_{\mathrm{eval}}(\theta_t - \epsilon g_B)
-
L_{\mathrm{eval}}(\theta_t).
$$

Negative $\Delta L$ means that the update reduced loss. The measured optimum is therefore

$$
\epsilon_{\mathrm{opt}}^{\mathrm{exp}}(B)
= \arg\min_{\epsilon}\, \overline{\Delta L}_B(\epsilon).
$$

We compare its dependence on batch size $B$ with the predicted form

$$
\epsilon_{\mathrm{opt}}^{\mathrm{theory}}(B)
= \frac{\epsilon_{\max}}{1 + B_{\mathrm{noise}} / B}.
$$

So the core question is: does the experimentally measured function $\epsilon_{\mathrm{opt}}(B)$ increase and saturate with batch size in the way predicted by the theory?

## Intuition: Why There Is an Optimal Step Size

Along a fixed update direction $g$, the local quadratic approximation gives the one-step loss change

$$
L(\theta - \epsilon g) - L(\theta)
\approx
-a\epsilon + \frac{1}{2}c\epsilon^2,
$$

where $a>0$ is the initial decrease rate and $c>0$ is the curvature along the update direction. The best one-step learning rate is

$$
\epsilon_{\mathrm{opt}} = \frac{a}{c}.
$$

The loss change first decreases, reaches its minimum at $\epsilon_{\mathrm{opt}}$, returns to zero at $2\epsilon_{\mathrm{opt}}$, and then becomes positive for larger steps. In the schematic plot, the x-axis is normalized as $\epsilon / \epsilon_{\mathrm{opt}}$.

![Quadratic learning-rate intuition](outputs/mnist_optimal_batch_lr_dense/quadratic_learning_rate_intuition.png)

## Results: 1. Finding the Maximum One-Step Loss Reduction

For each checkpoint and each batch size, the plot below shows the mean one-step loss change as a function of candidate learning rate $\epsilon$:

$$
\overline{\Delta L}_B(\epsilon)
= \frac{1}{K}\sum_{k=1}^{K}
\left[L_{\mathrm{eval}}(\theta_t - \epsilon g_{B,k}) - L_{\mathrm{eval}}(\theta_t)\right].
$$

The experimental optimum for each $B$ is the maximum loss reduction, equivalently the minimum of the $\Delta L$ curve because negative $\Delta L$ means loss went down. The axes are not normalized in this plot: x is raw $\epsilon$, and y is raw mean loss change. The vertical bars are standard errors across the sampled minibatches. The red-outlined colored `x` markers show the fitted-theory prediction $\epsilon_{\mathrm{opt}}^{\mathrm{theory}}(B)$ for the same batch size.

![Finding maximum one-step loss reduction](outputs/mnist_optimal_batch_lr_dense/combined_loss_change_vs_epsilon_with_theory.png)

## Results: 2. Normalized Experimental $\epsilon_{\mathrm{opt}}(B)$ vs Theoretical Trend

After finding the best epsilon for each batch size, we refine the grid optimum by fitting a local quadratic near the measured minimum. The theory fit uses these quadratic-refined optima, not the raw grid argmin, because the raw grid values are visibly discretized in the high-batch saturation regime.

We fit the measured points to

$$
\epsilon_{\mathrm{opt}}(B) = \frac{\epsilon_{\max}}{1 + B_{\mathrm{noise}} / B}.
$$

Then we normalize both axes using the fitted parameters:

$$
y = \frac{\epsilon_{\mathrm{opt}}}{\epsilon_{\max}},
\qquad
x = \frac{B}{B_{\mathrm{noise}}}.
$$

Under this normalization, the theoretical curve becomes

$$
y = \frac{x}{1+x}.
$$

The scatter points are the normalized quadratic-refined experimental optima. The line is the normalized theoretical trend.

![Normalized epsilon opt vs batch-noise theory comparison](outputs/mnist_optimal_batch_lr_dense/epsilon_opt_normalized_subplots.png)

## Why This Is Also a Batch-Size Result

The normalized $\epsilon_{\mathrm{opt}}(B)$ plot is not only about choosing the learning rate. Once the optimal learning rate depends on batch size as

$$
\epsilon_{\mathrm{opt}}(B)
=
\frac{\epsilon_{\max}}{1 + B_{\mathrm{noise}} / B},
$$

the local quadratic model also predicts the best achievable one-step loss change at that batch size. Using the standard convention $\Delta L = L_{\mathrm{after}} - L_{\mathrm{before}}$,

$$
\Delta L(\epsilon, B)
\approx
-\epsilon \lVert G \rVert^2
+
\frac{\epsilon^2}{2}
\left(G^T H G + \frac{\operatorname{tr}(H\Sigma)}{B}\right).
$$

At the optimum,

$$
\Delta L_{\mathrm{opt}}(B)
=
-\frac{1}{2}\lVert G \rVert^2\epsilon_{\mathrm{opt}}(B)
=
\frac{\Delta L_{\infty}}{1 + B_{\mathrm{noise}} / B}
=
\Delta L_{\infty}\frac{B}{B+B_{\mathrm{noise}}},
$$

where $\Delta L_{\infty}=-\frac{1}{2}\lVert G\rVert^2\epsilon_{\max}$ is the limiting optimal loss change for an exact full-batch gradient. Since this quantity is negative, the corresponding positive loss reduction is

$$
R_{\mathrm{opt}}(B)
=
-\Delta L_{\mathrm{opt}}(B)
=
R_{\infty}\frac{B}{B+B_{\mathrm{noise}}}.
$$

This is the batch-size consequence corresponding to Eq. 2.7 in the paper. Increasing batch size improves the best possible one-step progress, but only up to a saturation limit. For $B \ll B_{\mathrm{noise}}$, the positive loss reduction grows approximately linearly with $B$. Around $B \sim B_{\mathrm{noise}}$, the gains begin to slow. For $B \gg B_{\mathrm{noise}}$, the positive loss reduction is close to its maximum, so larger batches give diminishing additional progress per step.

### Coverage Above the Noise Scale

In this dense run, the largest tested batch size is `2048`, so the measured grid includes several points above the fitted $B_{\mathrm{noise}}$ scale. This gives a clearer view of whether $\epsilon_{\mathrm{opt}}/\epsilon_{\max}$ saturates toward `1` in the $B > B_{\mathrm{noise}}$ regime. The plotted optima use the quadratic-refined estimate because the raw grid optimum has only a few distinct values at high batch size and therefore looks artificially step-like.


## Fitted Parameters From Current Dense Run

| checkpoint | $\epsilon_{\max}$ | $B_{\mathrm{noise}}$ |
|---:|---:|---:|
| 100 | 0.148 | 49.8 |
| 500 | 0.187 | 151 |

These numbers are preliminary because the dense run used `K=10` sampled minibatches and a 2,000-example evaluation subset. A full run with larger `K` and the full evaluation subset should reduce noise in both the empirical optima and the fitted parameters.

## Secondary Check: $\Delta L_{\mathrm{opt}}$ vs $\epsilon_{\mathrm{opt}}$

The old plot of optimal loss change directly against batch size was not a good diagnostic because the expected relation is not easiest to see in those axes.

Under the local quadratic approximation,

$$
\overline{\Delta L}_B(\epsilon)
\approx
-\epsilon \lVert G \rVert^2
+ \frac{\epsilon^2}{2}\left(G^T H G + \frac{\operatorname{tr}(H\Sigma)}{B}\right).
$$

At the optimum,

$$
\Delta L_{\mathrm{opt}}(B)
\approx
-\frac{1}{2}\epsilon_{\mathrm{opt}}(B)\lVert G \rVert^2.
$$

So the clearer diagnostic is a linear-scale plot of $\Delta L_{\mathrm{opt}}$ against $\epsilon_{\mathrm{opt}}$, with a negative-slope line fit through the origin.

The fitted slope has a direct local-quadratic interpretation. Since

$$
\Delta L_{\mathrm{opt}}(B)
\approx
-\frac{1}{2}\epsilon_{\mathrm{opt}}(B)\lVert G \rVert^2,
$$

the through-origin slope should estimate

$$
-\frac{1}{2}\lVert G \rVert^2.
$$

Thus a more negative slope means a larger local full-gradient norm and more available one-step progress at that checkpoint. The much smaller magnitude at checkpoint 500 is consistent with being closer to convergence.

![Optimal loss change vs optimal epsilon](outputs/mnist_optimal_batch_lr_dense/loss_change_opt_vs_epsilon_opt_linear.png)

## Experimental Procedure

At checkpoint $\theta_t$ and batch size $B$:

1. Sample independent minibatches $b_k$.
2. Compute the minibatch gradient at the frozen checkpoint:

$$
g_{B,k} = \nabla_{\theta} L_{b_k}(\theta_t).
$$

3. For each candidate $\epsilon$, form the hypothetical update without mutating the checkpoint:

$$
\theta'_{B,k,\epsilon} = \theta_t - \epsilon g_{B,k}.
$$

4. Measure the evaluation loss change:

$$
\Delta L_{B,k}(\epsilon)
= L_{\mathrm{eval}}(\theta'_{B,k,\epsilon})
- L_{\mathrm{eval}}(\theta_t).
$$

5. Average over sampled minibatches:

$$
\overline{\Delta L}_B(\epsilon)
= \frac{1}{K}\sum_{k=1}^{K}\Delta L_{B,k}(\epsilon).
$$

6. Choose the empirical optimum:

$$
\epsilon_{\mathrm{opt}}^{\mathrm{exp}}(B)
= \arg\min_{\epsilon}\overline{\Delta L}_B(\epsilon).
$$

7. Fit the resulting $\left(B, \epsilon_{\mathrm{opt}}^{\mathrm{exp}}(B)\right)$ pairs to the theory curve.

## What Each Output File Means

- `raw_results.csv`: one row for each checkpoint, batch size, epsilon, and sampled minibatch.
- `aggregate_results.csv`: mean and standard error of standard `loss_change = L_after - L_before`; it also keeps legacy-compatible `delta_loss = L_before - L_after` with the opposite sign.
- `epsilon_opt_results.csv`: measured optimal epsilon for each checkpoint and batch size.
- `fit_results.csv`: fitted $\epsilon_{\max}$ and $B_{\mathrm{noise}}$.
- `training_loss_trace.csv`: fixed-evaluation-subset loss during the checkpoint-training run.
- `quadratic_learning_rate_intuition.png`: schematic local-quadratic picture explaining why an optimal one-step learning rate exists.
- `training_eval_loss_trace_checkpoints.png`: where checkpoints 100 and 500 occur relative to training loss and the cross-entropy lower bound.
- `combined_loss_change_vs_epsilon_with_theory.png`: how $\epsilon_{\mathrm{opt}}$ is computed from loss-change curves.
- `checkpoint_500_loss_change_near_zero_zoom.png`: zoomed checkpoint-500 loss-change curves near zero for representative batch sizes.
- `epsilon_opt_normalized_subplots.png`: normalized experimental-vs-theoretical comparison using $\epsilon_{\mathrm{opt}}/\epsilon_{\max}$ and $B/B_{\mathrm{noise}}$.
- `loss_change_opt_vs_epsilon_opt_linear.png`: secondary local-quadratic check that $\Delta L_{\mathrm{opt}}$ is approximately linear in $\epsilon_{\mathrm{opt}}$ with negative slope.
- `loss_change_opt_linear_fit.csv`: slopes for the through-origin linear fits in that secondary check.

## Reproducible Data Loading

In [ ]:
from pathlib import Path
import csv
import json

OUTPUT_DIR = Path("outputs/mnist_optimal_batch_lr_dense")

def read_csv(name):
    with (OUTPUT_DIR / name).open(newline="") as f:
        return list(csv.DictReader(f))

config = json.loads((OUTPUT_DIR / "config.json").read_text())
raw_rows = read_csv("raw_results.csv")
aggregate_rows = read_csv("aggregate_results.csv")
opt_rows = read_csv("epsilon_opt_results.csv")
fit_rows = read_csv("fit_results.csv")

print("raw rows:", len(raw_rows))
print("aggregate rows:", len(aggregate_rows))
print("epsilon-opt rows:", len(opt_rows))
fit_rows

In [ ]:
# Run parameters, including the epsilon grid used for this dense run.
config


## Reproduce the Retained Figures

The next cell regenerates the retained report figures from CSV outputs. It does not rerun training or measurement.

In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", str(Path(".matplotlib-cache").resolve()))
os.environ.setdefault("XDG_CACHE_HOME", str(Path(".cache").resolve()))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

def eps_model(B, epsilon_max, b_noise):
    B = np.asarray(B, dtype=float)
    return epsilon_max / (1.0 + b_noise / B)

steps = [int(s) for s in config["checkpoint_steps"]]
batches = [int(b) for b in config["batch_sizes"]]
fit_by_step = {int(r["checkpoint_step"]): r for r in fit_rows}
loss_curve_plot_batches = [4, 8, 16, 32, 64, 128, 256, 512]
loss_curve_plot_batches = [b for b in loss_curve_plot_batches if b in batches]
colors = plt.cm.tab10(np.linspace(0, 1, len(loss_curve_plot_batches)))

fig, axes = plt.subplots(1, len(steps), figsize=(7 * len(steps), 5), sharey=False)
if len(steps) == 1:
    axes = [axes]

for ax, step in zip(axes, steps):
    fit = fit_by_step[step]
    eps_max = float(fit["epsilon_max"])
    b_noise = float(fit["b_noise"])
    zoom_values = []
    local_region_epsilons = []

    for batch, color in zip(loss_curve_plot_batches, colors):
        rows = sorted(
            [r for r in aggregate_rows if int(r["checkpoint_step"]) == step and int(r["batch_size"]) == batch],
            key=lambda r: float(r["epsilon"]),
        )
        xs = np.array([float(r["epsilon"]) for r in rows])
        ys = np.array([float(r["mean_loss_change"]) for r in rows])
        yerr = np.array([float(r["stderr_loss_change"]) for r in rows])
        zoom_values.extend([y for y in ys if y < 0.2])
        local_region_epsilons.extend([x for x, y in zip(xs, ys) if y < 0.2])
        ax.errorbar(xs, ys, yerr=yerr, marker="o", linewidth=1.4, color=color, label=f"B={batch}")

        pred_eps = float(eps_model(batch, eps_max, b_noise))
        local_region_epsilons.append(pred_eps)
        pred_y = float(np.interp(pred_eps, xs, ys, left=ys[0], right=ys[-1]))
        ax.scatter([pred_eps], [pred_y], marker="x", s=130, linewidths=5.0, color="red", zorder=20)
        ax.scatter([pred_eps], [pred_y], marker="x", s=75, linewidths=2.2, color=color, zorder=21)

    if zoom_values:
        y_min, y_max = min(zoom_values), max(zoom_values)
        margin = 0.1 * max(y_max - y_min, 1e-3)
        ax.set_ylim(y_min - margin, y_max + margin)
    if local_region_epsilons:
        x_min, x_max = min(local_region_epsilons), max(local_region_epsilons)
        x_margin = 0.08 * max(x_max - x_min, 1e-3)
        ax.set_xlim(max(0.0, x_min - x_margin), x_max + x_margin)
    ax.set_title(f"checkpoint {step}")
    ax.set_xlabel("epsilon")
    ax.set_ylabel("mean loss change ΔL (lower = larger loss reduction; error bars: SE)")
    ax.grid(True, alpha=0.25)

axes[-1].legend(ncol=2, fontsize=8, loc="best")
fig.suptitle("Finding maximum one-step loss reduction on linear epsilon axes\nred-outlined colored x markers show fitted-theory epsilon_opt(B); optimum is minimum ΔL", y=1.05)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "combined_loss_change_vs_epsilon_with_theory.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(1, len(steps), figsize=(6.5 * len(steps), 5), sharey=True)
if len(steps) == 1:
    axes = [axes]

for ax, step in zip(axes, steps):
    rows = sorted([r for r in opt_rows if int(r["checkpoint_step"]) == step], key=lambda r: int(r["batch_size"]))
    B = np.array([int(r["batch_size"]) for r in rows], dtype=float)
    eps_exp = np.array([float(r["epsilon_opt_quadratic"]) for r in rows])
    fit = fit_by_step[step]
    eps_max = float(fit["epsilon_max"])
    b_noise = float(fit["b_noise"])
    x_fit = np.logspace(np.log2(B.min()), np.log2(B.max()), 300, base=2)

    x_exp = B / b_noise
    y_exp = eps_exp / eps_max
    x_fit = np.logspace(-2, 2, 300)
    y_fit = x_fit / (1.0 + x_fit)

    ax.scatter(x_exp, y_exp, s=55, label="quadratic-refined experimental optimum")
    ax.plot(x_fit, y_fit, linewidth=2.2, label="theory: x / (1 + x)")
    ax.set_xscale("log")
    ax.set_title(f"checkpoint {step}\nepsilon_max={eps_max:.3g}, B_noise={b_noise:.3g}")
    ax.set_xlabel("B / B_noise")
    ax.set_ylabel("epsilon_opt / epsilon_max")
    ax.grid(True, alpha=0.25, which="both")
    ax.legend(fontsize=8)

fig.suptitle("Normalized epsilon_opt(B) vs theoretical curve", y=1.03)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "epsilon_opt_normalized_subplots.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, axes = plt.subplots(1, len(steps), figsize=(6.5 * len(steps), 5), sharey=False)
if len(steps) == 1:
    axes = [axes]

linear_fit_rows = []
for ax, step in zip(axes, steps):
    rows = sorted([r for r in opt_rows if int(r["checkpoint_step"]) == step], key=lambda r: float(r["epsilon_opt_quadratic"]))
    epsilon_opt = np.array([float(r["epsilon_opt_quadratic"]) for r in rows])
    loss_change_opt = np.array([float(r["loss_change_opt_quadratic"]) for r in rows])
    slope = float(np.dot(epsilon_opt, loss_change_opt) / np.dot(epsilon_opt, epsilon_opt))
    x_line = np.linspace(0.0, max(epsilon_opt.max(), 1e-12), 200)
    ax.scatter(epsilon_opt, loss_change_opt, s=55, label="measured optima")
    ax.plot(x_line, slope * x_line, linewidth=2.0, label=f"through-origin fit: slope={slope:.3g}")
    ax.set_xlabel("epsilon_opt")
    ax.set_ylabel("optimal mean loss change")
    ax.set_title(f"checkpoint {step}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    linear_fit_rows.append((step, slope))

fig.suptitle("Optimal loss change vs optimal epsilon", y=1.03)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "loss_change_opt_vs_epsilon_opt_linear.png", dpi=180, bbox_inches="tight")
plt.close(fig)

print("wrote", OUTPUT_DIR / "combined_loss_change_vs_epsilon_with_theory.png")
print("wrote", OUTPUT_DIR / "epsilon_opt_normalized_subplots.png")
print("wrote", OUTPUT_DIR / "loss_change_opt_vs_epsilon_opt_linear.png")

## Supplemental: Training Context for the Checkpoints

Checkpoint 100 and checkpoint 500 correspond to different local regimes. Step 100 is still in a part of training where one-step updates can noticeably reduce loss. Step 500 is closer to the low-loss plateau, so there is much less remaining loss reduction available. The dashed line marks the theoretical lower bound of cross-entropy loss, $0$.

![Training loss trace with checkpoints](outputs/mnist_optimal_batch_lr_dense/training_eval_loss_trace_checkpoints.png)

## Supplemental: Checkpoint 500 Near-Zero Zoom

The checkpoint-500 loss-change curves are almost flat near zero before rising above zero at larger $\epsilon$. This is consistent with the model being closer to convergence: there are still small negative one-step loss changes, especially for larger batches, but the scale is much smaller than at checkpoint 100 and too-large steps quickly increase loss. This supplemental zoom shows representative batch sizes only to keep the near-zero curvature readable. Red-outlined colored `x` markers show the fitted-theory $\epsilon_{\mathrm{opt}}(B)$ values on the same curves.

![Checkpoint 500 near-zero zoom](outputs/mnist_optimal_batch_lr_dense/checkpoint_500_loss_change_near_zero_zoom.png)